In [1]:
import numpy as np
import pandas as pd
import os
import re
from datetime import datetime
       

 # The star of the show
from google_play_scraper import app, reviews, Sort      

import sys,os 
sys.path.insert(0, os.path.abspath('..'))

---
## 1. Scraping Reviews

Now let's collect the actual reviews. The `reviews()` function lets us specify:
- **How many** reviews to fetch (`count`)
- **Which order** to sort them (`Sort.NEWEST` or `Sort.MOST_RELEVANT`)
- **Filter by rating** (we want all ratings, so `None`)


In [2]:
from src.data_scrapping import scrape_fintech_reviews, APPS, clean_text

df_raw = scrape_fintech_reviews(APPS, count=500)
df_raw = pd.DataFrame(df_raw)
print(f"Shape: {df_raw.shape}")
df_raw.head()

  Scraped 500 reviews for CBE Mobile
  Scraped 500 reviews for BoA Mobile
  Scraped 500 reviews for Dashen Mobile
Shape: (1500, 8)


,review_id,app,review,rating,date,thumbs,bank,source
0,74bf72d1-d6ce-4bd7-9a4d-3950846e5aba,CBE Mobile,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,5,2026-05-14 12:13:13,0,CBE Mobile,Google Play
1,0e691771-0024-4326-b71f-1685b91dc83d,CBE Mobile,incredible,5,2026-05-14 10:53:56,0,CBE Mobile,Google Play
2,982b1262-3d54-4a12-811a-8b26b7ecc777,CBE Mobile,best app for financial sector,5,2026-05-14 05:44:01,0,CBE Mobile,Google Play
3,06f6640c-b65c-43e4-88ef-0a79be8b9534,CBE Mobile,it's a good application,5,2026-05-13 20:28:58,0,CBE Mobile,Google Play
4,eda74236-f7f3-4422-a139-a181e832bc27,CBE Mobile,thank you cbe,5,2026-05-13 17:16:37,0,CBE Mobile,Google Play


In [3]:
os.makedirs('../data/raw', exist_ok=True)

output_path = '../data/raw/three_banks_raw1.csv'
df_raw.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/raw/three_banks_raw1.csv


In [4]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  940  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  4 stars:  111  ██████████████████████
  3 stars:   80  ████████████████
  2 stars:   48  █████████
  1 stars:  321  ████████████████████████████████████████████████████████████████


---
## 2. EDA and Data Quality Audit

> **"Garbage In, Garbage Out (GIGO)"** — If our input data is flawed, every insight we build on top of it is also flawed.

Let's audit our three common data quality problems:
1. Missing values
2. Duplicate reviews  
3. Inconsistent date formats

In [5]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(3).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

Sample date values (raw):
0   2026-05-14 12:13:13
1   2026-05-14 10:53:56
2   2026-05-14 05:44:01

Date dtype: datetime64[ns]
DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  app            : OK
  review         : OK
  rating         : OK
  date           : OK
  thumbs         : OK
  bank           : OK
  source         : OK


In [6]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")


Problem 2: Duplicates
------------------------------
  Duplicate review IDs    : 0


In [7]:
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")
print(df_raw.head(3))

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[ns]
  Sample values: 2026-05-14 12:13:13
  Target format: YYYY-MM-DD (string or date object)
                              review_id         app  \
0  74bf72d1-d6ce-4bd7-9a4d-3950846e5aba  CBE Mobile   
1  0e691771-0024-4326-b71f-1685b91dc83d  CBE Mobile   
2  982b1262-3d54-4a12-811a-8b26b7ecc777  CBE Mobile   

                          review  rating                date  thumbs  \
0      yoroo namaste 🙏 ♥️ ❤️ 💖 💖       5 2026-05-14 12:13:13       0   
1                     incredible       5 2026-05-14 10:53:56       0   
2  best app for financial sector       5 2026-05-14 05:44:01       0   

         bank       source  
0  CBE Mobile  Google Play  
1  CBE Mobile  Google Play  
2  CBE Mobile  Google Play  


---
## 3. Cleaning Strategy

Now we fix each problem, one at a time.  
We work on a **copy** so we can always compare back to the raw data.

### Strategy Overview

| Problem | Strategy | Tool |
|---------|----------|------|
| Missing critical data | Drop the row | `df.dropna()` |
| Duplicate reviews | Keep first occurrence | `df.drop_duplicates()` |
| Inconsistent dates | Parse and reformat | `pd.to_datetime()` |
| Messy text | Strip whitespace / remove junk | `str.strip()`, `re.sub()` |

In [8]:
# Work on a copy so raw data stays untouched
df = df_raw.copy()
print(df_raw.head(3))
print(f"Starting with: {len(df)} reviews")

                              review_id         app  \
0  74bf72d1-d6ce-4bd7-9a4d-3950846e5aba  CBE Mobile   
1  0e691771-0024-4326-b71f-1685b91dc83d  CBE Mobile   
2  982b1262-3d54-4a12-811a-8b26b7ecc777  CBE Mobile   

                          review  rating                date  thumbs  \
0      yoroo namaste 🙏 ♥️ ❤️ 💖 💖       5 2026-05-14 12:13:13       0   
1                     incredible       5 2026-05-14 10:53:56       0   
2  best app for financial sector       5 2026-05-14 05:44:01       0   

         bank       source  
0  CBE Mobile  Google Play  
1  CBE Mobile  Google Play  
2  CBE Mobile  Google Play  
Starting with: 1500 reviews


### 3.1 Handle Missing Values

The **review text** and **rating** are critical — without them, the row is useless for analysis.  
We drop any row missing these. Other fields (like `review_id`) are less critical.

In [9]:
missing_review = df['review'].isna().sum()
missing_rating = df['rating'].isna().sum()

before_dropna = len(df)

df = df.dropna(subset=['review', 'rating'])


after_dropna = len(df)
dropped = before_dropna - after_dropna
print(f"\n2. Missing values handled:")
print(f"   - Rows with missing review text: {missing_review}")
print(f"   - Total rows dropped: {dropped}")
print(f"   Shape after dropna: {after_dropna}")




2. Missing values handled:
   - Rows with missing review text: 0
   - Total rows dropped: 0
   Shape after dropna: 1500


### 3.2 Remove duplicate

In [10]:
before_dedup = len(df)
df = df.drop_duplicates(
    subset=['review', 'rating', 'date', 'bank'], 
    keep='first'
).copy()
after_dedup = len(df)
print(f"\n1. Duplicates removed: {before_dedup - after_dedup} rows")
print(f"   Shape before dedup: {before_dedup}")
print(f"   Shape after dedup: {after_dedup}")


1. Duplicates removed: 0 rows
   Shape before dedup: 1500
   Shape after dedup: 1500


### 3.3 Normalize dates
Dates need to be in a consistent `YYYY-MM-DD` format for:
- Time-series analysis (sentiment over time)
- Sorting
- Database storage

The `date` field from the scraper is a Python `datetime` object. We convert it to a clean date string.

we have date range :
Date range: 2025-02-12 to 2026-05-13



In [11]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
print(f"\n3. Dates normalized to YYYY-MM-DD format")
print(df['date'].head(3))

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-05-14 12:13:13
1   2026-05-14 10:53:56
2   2026-05-14 05:44:01
dtype: datetime64[ns]

3. Dates normalized to YYYY-MM-DD format
0    2026-05-14
1    2026-05-14
2    2026-05-14
Name: date, dtype: object

Date range: 2025-02-12 to 2026-05-14


### 3.4 Cleaning Review tests
- remove whitespaces and newlines
- empty strings and extraspaces

In [12]:
# Show before/after on a sample review
sample_raw = "       yoroo namaste 🙏 ♥️ ❤️ 💖 💖"
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)
print(f'first three cleaned reviews:')
print(df['review'].head(3))

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")


Before: '       yoroo namaste 🙏 ♥️ ❤️ 💖 💖'
After : 'yoroo namaste'
first three cleaned reviews:
0                    yoroo namaste
1                       incredible
2    best app for financial sector
Name: review, dtype: object

Removed 78 reviews that were empty after cleaning


### 3.5 Valid ratings
- ratings should be whole numbers from 1 - 5

In [13]:
# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 1422 reviews
Rating dtype: int32


# Out Put cleaned and processed data to new data set

Now we produce the clean, standardized dataset with **exactly the 5 required columns**:  
`review`, `rating`, `date`, `bank`, `source`

This is the dataset that feeds into Task 2 (Sentiment Analysis) and Task 3 (Database).

In [14]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (1422, 5)


,review,rating,date,bank,source
0,yoroo namaste,5,2026-05-14,CBE Mobile,Google Play
1,incredible,5,2026-05-14,CBE Mobile,Google Play
2,best app for financial sector,5,2026-05-14,CBE Mobile,Google Play
3,i swear to god by using this app i won a sa...,5,2026-05-14,Dashen Mobile,Google Play
4,good,5,2026-05-14,Dashen Mobile,Google Play
5,good and easier to used,5,2026-05-14,Dashen Mobile,Google Play
6,very nice app,5,2026-05-13,Dashen Mobile,Google Play
7,very difficult app,1,2026-05-13,Dashen Mobile,Google Play
8,good app but debit transactions not allowed why,5,2026-05-13,Dashen Mobile,Google Play
9,nice,5,2026-05-13,Dashen Mobile,Google Play


In [15]:
os.makedirs('../data/processed', exist_ok=True)

output_path = '../data/processed/three_banks_cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/processed/three_banks_cleaned.csv


# report


In [16]:
print("=" * 55)
print("  PREPROCESSING REPORT — Results of the three banks dataset")
print("=" * 55)

original_count = len(df_raw)
final_count    = len(df_clean)
removed_total  = original_count - final_count
retention_rate = (final_count / original_count * 100)

print(f"\n  Raw reviews collected  : {original_count:>6}")
print(f"  Reviews after cleaning : {final_count:>6}")
print(f"  Reviews removed        : {removed_total:>6}")
print(f"  Data retention rate    : {retention_rate:>5.1f}%")

quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
print(f"  Data quality           : {quality}")

print(f"\n  Date range : {df_clean['date'].min()}  to  {df_clean['date'].max()}")

print("\n  Rating distribution:")
for rating in sorted(df_clean['rating'].unique(), reverse=True):
    count = (df_clean['rating'] == rating).sum()
    pct   = count / final_count * 100
    bar   = '█' * (count // 5)
    print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

print("\n  Text length stats:")
lengths = df_clean['review'].str.len()
print(f"    Min    : {lengths.min()} characters")
print(f"    Median : {lengths.median():.0f} characters")
print(f"    Max    : {lengths.max()} characters")

print("\n  Columns in final CSV:")
for col in df_clean.columns:
    print(f"    - {col}")

print("\n" + "=" * 55)

  PREPROCESSING REPORT — Results of the three banks dataset

  Raw reviews collected  :   1500
  Reviews after cleaning :   1422
  Reviews removed        :     78
  Data retention rate    :  94.8%
  Data quality           : GOOD

  Date range : 2025-02-12  to  2026-05-14

  Rating distribution:
    5 stars :  893 (62.8%)  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
    4 stars :  101 ( 7.1%)  ████████████████████
    3 stars :   76 ( 5.3%)  ███████████████
    2 stars :   46 ( 3.2%)  █████████
    1 stars :  306 (21.5%)  █████████████████████████████████████████████████████████████

  Text length stats:
    Min    : 1 characters
    Median : 16 characters
    Max    : 499 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

